# Chapter 3 - Classification

In [4]:
import sys
sys.path.append("../")
from utils import *

In [ ]:
def sample_beta(limit1, limit2):
  margin1 = limit1 + (limit2 - limit1)*0.45
  margin2 = limit2 - (limit2 - limit1)*0.45
  beta = np.random.uniform(margin1, margin2)
  return beta

def create_data_bagging(d = 4, number_of_members = 1, n_samples = 1000):
  # Creates n samples
  samples = np.random.uniform(size=(n_samples, 2))

  samples_of_half = "samples_of_half"
  x_1 = "x_1"; x_2 = "x_2"; y_1 = "y_1"; y_2 = "y_2"; tag = "tag"
  list_of_array = {0: {samples_of_half : samples, x_1 : 0, x_2 : 1, y_1 : 0, y_2 : 1}}

  for i in range(0, d):
    built_list =  {}
    for sample_curr_i, sample_curr in enumerate(list_of_array.values()):
      # Choose if we want to split x axis or y axis
      dim_half = np.random.choice([0,1])

      dots_coords = sample_curr[samples_of_half]
      if (dim_half == 0):
        beta = sample_beta(sample_curr[x_1], sample_curr[x_2])
        built_list[sample_curr_i*2] = {samples_of_half: dots_coords[dots_coords[:,0] <= beta],
                                       x_1 : sample_curr[x_1],
                                       x_2 : beta,
                                       y_1 : sample_curr[y_1],
                                       y_2 : sample_curr[y_2],
                                       tag : np.random.choice([0, 1]).astype(int)}

        built_list[sample_curr_i*2 + 1] = {samples_of_half: dots_coords[dots_coords[:,0] > beta],
                                       x_1 : beta,
                                       x_2 : sample_curr[x_2],
                                       y_1 : sample_curr[y_1],
                                       y_2 : sample_curr[y_2],
                                       tag : np.random.choice([0, 1]).astype(int)}
      else:
        beta = sample_beta(sample_curr[y_1], sample_curr[y_2])
        built_list[sample_curr_i*2] = {samples_of_half: dots_coords[dots_coords[:,1] <= beta],
                                       x_1 : sample_curr[x_1],
                                       x_2 : sample_curr[x_2],
                                       y_1 : sample_curr[y_1],
                                       y_2 : beta,
                                       tag : np.random.choice([0, 1]).astype(int)}

        built_list[sample_curr_i*2 + 1] = {samples_of_half: dots_coords[dots_coords[:,1] > beta],
                                       x_1 : sample_curr[x_1],
                                       x_2 : sample_curr[x_2],
                                       y_1 : beta,
                                       y_2 : sample_curr[y_2],
                                       tag : np.random.choice([0, 1]).astype(int)}


    list_of_array =  built_list
  return built_list

n_train = 500
n_test = 500

built_list = create_data_bagging(n_samples = n_train + n_test, d = 6)
reorder_samples = np.arange(n_train + n_test)
np.random.shuffle(reorder_samples)
reorder_samples = reorder_samples.tolist()
samples_all = np.vstack([samples_["samples_of_half"] for samples_ in built_list.values()])
tags_all = np.hstack([np.repeat(samples_["tag"], samples_["samples_of_half"].shape[0]) for samples_ in built_list.values()])

samples, tags = samples_all[reorder_samples[:n_train],:], tags_all[reorder_samples[:n_train]]
samples_test, tags_test = samples_all[reorder_samples[n_train:],:], tags_all[reorder_samples[n_train:]]

depths = list(range(1, 16))
clf_lst = [tree.DecisionTreeClassifier(max_depth=depth, random_state=42).fit(samples, tags) for depth in depths]
train_predictions = [clf.predict(samples) for clf in clf_lst]
train_error = 1 - np.sum(train_predictions == tags, axis = 1)/len(tags)

test_predictions = [clf.predict(samples_test) for clf in clf_lst]
test_error = 1 - np.sum(test_predictions == tags_test, axis = 1)/len(tags_test)

x_range = (np.array(list(range(0, 101, 2)))/100).tolist()
y_range = (np.array(list(range(0, 101, 2)))/100).tolist()
xx, yy = np.meshgrid(x_range,y_range)
xx = xx.ravel()
yy = yy.ravel()

meshgrid_predictions = [clf.predict(np.vstack([xx, yy]).T) for clf in clf_lst]

fig = make_subplots(
    rows=1, cols=2, subplot_titles=('Train and Test Error', 'Predictions - Decisions boundaries'),
    horizontal_spacing=0.08,   column_widths=[0.42, 0.5]
)

frames = [go.Frame(name=depth_trees,
               data=[go.Scatter(x= xx,  y= yy, mode='markers', marker = dict(symbol = "square", color = meshgrid_predictions[i]), opacity = 0.1, showlegend=False),
                     go.Scatter(x= samples_test[:,0],  y= samples_test[:,1],mode='markers', marker = dict(color = tags_test), showlegend=False),
                     go.Scatter(x = np.array(depths)[:depth_trees], y=np.array(train_error)[:depth_trees], name = "Train error"),
                     go.Scatter(x = np.array(depths)[:depth_trees], y=np.array(test_error)[:depth_trees], name = "Test error"),],
                     #go.Scatter(x = np.array(depths)[:depth_trees], y=np.array(test_error)[:depth_trees])],
               traces=[0,1,2,3]) for i, depth_trees in enumerate(depths)]


fig.add_traces(data=frames[0]["data"], rows=[1,1,1,1], cols=[1,1,2,2])
fig.frames=frames
button = dict(
             label='Play',
             method='animate',
             args=[None, dict(frame=dict(duration=500, redraw=True),
                              transition=dict(duration=0),
                              easing='linear',
                              fromcurrent=True,
                              redraw = True,
                              mode='immediate')])
fig.update_layout(updatemenus=[dict(type='buttons',
                              showactive=False,
                              y=0,
                              x=1.05,
                              xanchor='left',
                              yanchor='bottom',
                              buttons=[button] )
                                      ],
                 width=1200, height=500)

fig.update_yaxes(title_text="Error Rate", range=[-0.05, 0.6], row=1, col=2)
fig.update_xaxes(title_text="Depth of tree", range=[depths[0], depths[-1]], row=1, col=2)


fig.update_yaxes(title_text="", range=[0, 1], row=1, col=1)
fig.update_xaxes(title_text="", range=[0, 1], row=1, col=1)
iplot(fig)

In [32]:
def decision_surface(model, xrange, yrange):
    xx, yy = np.meshgrid(xrange, yrange)
    pred = model.predict(np.c_[xx.ravel(), yy.ravel()])
    print(pred)
    return go.Contour(x=xrange, y=yrange, z=pred, showscale=False, colorscale='RdBu', opacity=0.4, name='Score', hoverinfo='skip')


## Batch Perceptron

In [34]:
class Perceptron:
    def __init__(self, tol=1e-2, max_iter=100):
        self.w = None
        self.tol = tol
        self.max_iter = max_iter
        
    def fit(self, X: np.ndarray, y: np.ndarray):
        self.w, max_iter = [np.zeros(X.shape[1])], self.max_iter

        while max_iter > 0:
            for i in range(X.shape[0]):
                if y[i] != np.sign(X[i] @ self.w[-1]):
                    self.w.append(self.w[-1] + y[i] * X[i])
                    if np.linalg.norm(self.w[-1] - self.w[-2]) <= self.tol:
                        return self
            max_iter -= 1
        return self
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        assert self.w, "Model must first be trained before calling `predict`"
        return np.sign(X @ self.w[-1], axis=0)
    
    def fit_predict(self, X: np.ndarray, y: np.ndarray) -> np.ndarray:
        return self.fit(X, y).predict(X)

In [35]:
n = 10
X = np.concatenate([np.random.multivariate_normal([0,0],[[1,0],[0,1]], n), 
                    np.random.multivariate_normal([5,5],[[1,0],[0,1]], n)])
y = np.concatenate([np.ones(n), -np.ones(n)])


perceptron = Perceptron().fit(X, y)

go.Figure([go.Scatter(x=X[:n,0], y=X[:n,1], mode="markers", marker=dict(color="blue")),
           go.Scatter(x=X[n:,0], y=X[n:,1], mode="markers", marker=dict(color="red")),
           decision_surface(perceptron, 
                            np.arange(X[:,0].min(), X[:,0].max(), 100),
                            np.arange(X[:,1].min(), X[:,1].max(), 100))]).show()





TypeError: 'axis' is an invalid keyword to ufunc 'sign'

# AdaBoost Training Process

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier


class StagedAdaBoostClassifier(AdaBoostClassifier):
    def __init__(self, **kwargs):
        super().__init__(*kwargs)
        self.sample_weights = []

    def _boost(self, iboost, X, y, sample_weight, random_state):
        res = super()._boost(iboost, X, y, sample_weight, random_state)
        self._iteration_callback(iboost, X, y, *res)
        return res

    def _iteration_callback(self, iboost, X, y, sample_weight, estimator_weight, estimator_error):
        self.sample_weights.append(sample_weight)


In [ ]:
# X = np.array([[x1, x2] for x1 in range(10) for x2 in range(10)])
# y = np.zeros(len(X))

# y[np.where((3<=X[:,0]) & (X[:,0]<6) & (3<=X[:,1]) & (X[:,1]<6))] = 1




# X = np.array([[4,8,5.5,5,1,6,4,2,5,9],
#               [1,1,2,3,5.5,5,6,7,8,7]]).T
# y=np.array([1,0,1,1,1,0,0,1,0,0])


[np.random.normal([1,1],[[1,0],[0,1]], 10)]



In [ ]:
import plotly.graph_objects as go
import numpy as np

model = StagedAdaBoostClassifier().fit(X,y)

frames = []
for i, D in enumerate([np.repeat(1/len(X), len(X))] + model.sample_weights):
    #D /= D.sum()
    frames.append(go.Frame(data=[go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers",
                                            marker=dict(color=y, size=D*100))],
                           layout=go.Layout(title_text="Iteration " + str(i) )))

go.Figure(data=frames[0]["data"],
          frames=frames[1:],
          layout=go.Layout(
              title=frames[0]["layout"]["title"],
              updatemenus=[dict(visible=True,
                              type="buttons",
                              buttons=[dict(label="Play",
                                            method="animate",
                                            args=[None, dict(frame={"duration":1000}) ])])]  )).show()

In [3]:
import plotly.graph_objects as go
import numpy as np

model = StagedAdaBoostClassifier().fit(X,y)

frames = []
for i, D in enumerate([np.repeat(1/len(X), len(X))] + model.sample_weights):
    #D /= D.sum()
    frames.append(go.Frame(data=[go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers",
                                            marker=dict(color=y, size=D*100))],
                           layout=go.Layout(title_text="Iteration " + str(i) )))

go.Figure(data=frames[0]["data"],
          frames=frames[1:],
          layout=go.Layout(
              title=frames[0]["layout"]["title"],
              updatemenus=[dict(visible=True,
                              type="buttons",
                              buttons=[dict(label="Play",
                                            method="animate",
                                            args=[None, dict(frame={"duration":1000}) ])])]  )).show()

In [80]:
# X = np.array([[x1, x2] for x1 in range(10) for x2 in range(10)])
# y = np.zeros(len(X))

# y[np.where((3<=X[:,0]) & (X[:,0]<6) & (3<=X[:,1]) & (X[:,1]<6))] = 1




# X = np.array([[4,8,5.5,5,1,6,4,2,5,9],
#               [1,1,2,3,5.5,5,6,7,8,7]]).T
# y=np.array([1,0,1,1,1,0,0,1,0,0])


[np.random.normal([1,1],[[1,0],[0,1]], 10)]



ValueError: shape mismatch: objects cannot be broadcast to a single shape

In [78]:
import plotly.graph_objects as go
import numpy as np

model = StagedAdaBoostClassifier().fit(X,y)

frames = []
for i, D in enumerate([np.repeat(1/len(X), len(X))] + model.sample_weights):
    #D /= D.sum()
    frames.append(go.Frame(data=[go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers",
                                            marker=dict(color=y, size=D*100))],
                           layout=go.Layout(title_text="Iteration " + str(i) )))

go.Figure(data=frames[0]["data"],
          frames=frames[1:],
          layout=go.Layout(
              title=frames[0]["layout"]["title"],
              updatemenus=[dict(visible=True,
                              type="buttons",
                              buttons=[dict(label="Play",
                                            method="animate",
                                            args=[None, dict(frame={"duration":1000}) ])])]  )).show()